In [ ]:
import os
import argparse
from glob import glob
from time import time
from PIL import Image, ImageDraw
from pathlib import Path
from IPython.display import display


import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection, CLIPImageProcessor, Owlv2ImageProcessor, Owlv2ImageProcessorFast

from time import time
import matplotlib.pyplot as plt

from scipy import ndimage
STANDARD_SIZE = {"height": 960, "width": 960}

In [ ]:
# img_paths = pd.Series(sorted(glob("../TTP_images/*.jpg")))
img_paths = pd.Series(sorted(glob("/mnt/TTP/images2/*.jpg")))

terms = pd.read_csv("./object_terms.csv")
tags = list(terms["label_en"])

def load_imgs(paths):
    loaded = []
    for f in paths:
        try:
            img = Image.open(f)  # .convert("RGB")
            img.load()
            loaded.append(img.convert("RGB"))
        except Exception as e:
            print(f"Loader failed on path {f}, {e}")
    return loaded

rand_paths = img_paths.sample(20)
imgs = load_imgs(rand_paths)

---
## basic pipeline

In [ ]:
owl2 = "google/owlv2-base-patch16"
auto_proc = AutoProcessor.from_pretrained(owl2)
# owl_proc = Owlv2ImageProcessor(use_fast=True)
# fast_proc = Owlv2ImageProcessorFast()
model = AutoModelForZeroShotObjectDetection.from_pretrained(owl2)

In [ ]:
t0 = time()
# resized = [i.resize((960, 960)) for i in imgs]
imgs_for_proc = imgs[1:5]
owl_inputs = auto_proc(text=[tags]*len(imgs_for_proc), images=imgs_for_proc, return_tensors="pt")
t1 = time()-t0
print(f"that took {t1:.2f}s ({t1/len(imgs_for_proc):.2f}s per image)")

In [ ]:
t0 = time()
with torch.no_grad():
    outputs = model(**owl_inputs)
t1 = time()-t0
print(f"that took {t1:.2f}s ({t1/len(imgs_for_proc):.2f}s per image)")

In [ ]:
# detections = auto_proc.post_process_grounded_object_detection(
#             outputs,
#             threshold=0.1,
#             target_sizes=[i.size for i in imgs_for_proc],
#             text_labels=[tags]*len(imgs_for_proc),
# )

In [ ]:
for i in imgs_for_proc:
    display(i)

In [ ]:
auto_proc.post_process_grounded_object_detection(
   outputs, threshold=0.4, target_sizes=[i.size for i in imgs_for_proc], text_labels=[tags]*len(imgs_for_proc)

)

In [ ]:

results = auto_proc.post_process_grounded_object_detection(
   outputs, threshold=0.2, target_sizes=[i.size for i in imgs_for_proc], text_labels=[tags]*len(imgs_for_proc)

)

for i, r in zip(imgs_for_proc, results):
    draw_img = i.copy()
    draw = ImageDraw.Draw(draw_img)
    
    scores = r["scores"]
    text_labels = r["text_labels"]
    boxes = r["boxes"]
    
    for box, score, text_label in zip(boxes, scores, text_labels):
        xmin, ymin, xmax, ymax = box
        draw.rectangle((xmin, ymin, xmax, ymax), outline="red", width=1)
        draw.text((xmin, ymin), f"{text_label}: {round(score.item(),2)}", fill="white")
    
    display(draw_img)

In [ ]:
results

In [ ]:
detections

In [ ]:
imgs_for_proc[2]

---
## pipeline from huggingface

In [ ]:
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
from transformers.image_utils import load_image
owl2 = "google/owlv2-base-patch16"
model = AutoModelForZeroShotObjectDetection.from_pretrained(owl2)
processor = AutoProcessor.from_pretrained(owl2, do_resize=True)
fast_proc = Owlv2ImageProcessorFast()

model.eval()

In [ ]:
# imgs_for_proc = imgs[:5]
url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/zero-sh-obj-detection_3.png"
# imgs_for_proc = [load_image(url).resize((960, 960))]
imgs_for_proc = [i.resize((960, 960)) for i in imgs[:5]]
text_labels = [tags]*len(imgs_for_proc)
t0 = time()

 #, do_rescale=False, do_pad=False, do_resize=False)
inputs = processor(images=imgs_for_proc, return_tensors="pt", do_resize=True) |\
        processor(text=text_labels, return_tensors="pt")
t1 = time()-t0
print(f"that took {t1:.2f}s ({t1/len(imgs_for_proc):.2f}s per image)")

In [ ]:
t0 = time()
with torch.inference_mode():
    outputs = model(**inputs)

t1 = time()-t0
print(f"that took {t1:.2f}s ({t1/len(imgs_for_proc):.2f}s per image)")

In [ ]:
results = processor.post_process_grounded_object_detection(
   outputs, threshold=0.3, target_sizes=[i.size for i in imgs_for_proc], text_labels=text_labels

)

for i, r in zip(imgs_for_proc, results):
    draw_img = Image.fromarray(np.asarray(i).copy()).resize((960, 960))
    draw = ImageDraw.Draw(draw_img)
    
    for box, score, text_label in zip(r["boxes"], r["scores"], r["text_labels"]):
        xmin, ymin, xmax, ymax = box
        draw.rectangle((xmin, ymin, xmax, ymax), outline="red", width=1)
        draw.text((xmin, ymin), f"{text_label}: {round(score.item(),2)}", fill="white")
    
    display(draw_img)

In [ ]:
imgs[4].resize((960, 960))

In [ ]:
img_paths = pd.Series(sorted(glob("/mnt/TTP/images1/*.jpg")))
o1 = pd.read_csv("../outputs/object_tagging/object_scores_images1.csv")

In [ ]:
rand = o1.sample(1)
rand

In [ ]:
rand_img = Image.open("/mnt/TTP/images1/"+rand.iloc[0].filename)

imgs_for_proc = [rand_img]#.resize((960, 960))]
text_labels = [tags]*len(imgs_for_proc)
t0 = time()

 #, do_rescale=False, do_pad=False, do_resize=False)
inputs = processor(images=imgs_for_proc, return_tensors="pt", do_resize=True) |\
        processor(text=text_labels, return_tensors="pt")
t1 = time()-t0
print(f"that took {t1:.2f}s ({t1/len(imgs_for_proc):.2f}s per image)")

In [ ]:
t0 = time()
with torch.inference_mode():
    outputs = model(**inputs)

t1 = time()-t0
print(f"that took {t1:.2f}s ({t1/len(imgs_for_proc):.2f}s per image)")

In [ ]:
results = processor.post_process_grounded_object_detection(
   outputs, threshold=0.01, target_sizes=[i.size for i in imgs_for_proc], text_labels=text_labels

)

for i, r in zip(imgs_for_proc, results):
    draw_img = Image.fromarray(np.asarray(i).copy())#.resize((960, 960))
    draw = ImageDraw.Draw(draw_img)
    
    for box, score, text_label in zip(r["boxes"], r["scores"], r["text_labels"]):
        xmin, ymin, xmax, ymax = box
        draw.rectangle((xmin, ymin, xmax, ymax), outline="red", width=1)
        draw.text((xmin, ymin), f"{text_label}: {round(score.item(),2)}", fill="white")
    
    display(draw_img)

In [ ]:
files = list(rand.filename)
files

In [ ]:
overall = []
recs = []
for f, cur in zip(files, results):
    zipped = zip(
        cur["scores"],
                cur["boxes"],
                cur["text_labels"],
                cur["labels"],
                range(13),
            )
    for s, b, l_, l_id, _ in zipped:
        recs.append([f, round(float(s), 3), *map(int, b), l_, int(l_id)])

    overall.extend(recs)
    # time_stats["recs"].append(time() - start)


In [ ]:
pd.DataFrame([results[0]["scores"].numpy()[0].round(3)]).values

In [ ]:
cur = [*results[0]["boxes"].numpy().astype(int)[0]]
pd.DataFrame([cur]).iloc[0, 0]

---
## compare

In [ ]:
test_result = pd.read_csv("./test_output.csv")

test_result.filename.unique()

In [ ]:
test_img = Image.open("/mnt/TTP/images1/"+"TM-10051448.jpg")
test_img2 = Image.open("/mnt/TTP/images1/"+"TM-10060344.jpg")
imgs_for_proc = [test_img.resize((960, 960)), test_img2.resize((960, 960))]
# imgs_for_proc = [test_img, test_img2]
text_labels = [tags]*len(imgs_for_proc)

t0 = time()
 #, do_rescale=False, do_pad=False, do_resize=False)
nb_inputs = processor(images=imgs_for_proc, return_tensors="pt", do_resize=True) |\
        processor(text=text_labels, return_tensors="pt")
t1 = time()-t0
print(f"that took {t1:.2f}s ({t1/len(imgs_for_proc):.2f}s per image)")

model.eval()

t0 = time()
with torch.inference_mode():
    nb_outputs = model(**nb_inputs)

t1 = time()-t0
print(f"that took {t1:.2f}s ({t1/len(imgs_for_proc):.2f}s per image)")

In [ ]:
results = processor.post_process_grounded_object_detection(
   nb_outputs, threshold=0.2, target_sizes=[i.size for i in imgs_for_proc], text_labels=text_labels
)

for i, r in zip(imgs_for_proc, results):
    draw_img = Image.fromarray(np.asarray(i).copy()).resize((960, 960))
    draw = ImageDraw.Draw(draw_img)
    
    for box, score, text_label in zip(r["boxes"], r["scores"], r["text_labels"]):
        xmin, ymin, xmax, ymax = box
        draw.rectangle((xmin, ymin, xmax, ymax), outline="red", width=1)
        draw.text((xmin, ymin), f"{text_label}: {round(score.item(),2)}", fill="white", textsize=20)
    
    display(draw_img)

In [ ]:
for f, cur in zip(["a", "b"], results):
    zipped = zip(
                cur["scores"].numpy().round(3),
                cur["boxes"].numpy().astype(int),
                cur["text_labels"],
                cur["labels"],
            )
    zipped = zip(sorted(zipped, key=lambda t: t[0], reverse=True), range(20))
    for (s, b, l_, l_id), i in zipped:
        print([f, s, *b, l_, i])
    print()


In [ ]:
test_result[test_result.filename == "TM-10051448.jpg"]

In [ ]:
Image.open("/mnt/TTP/images1/"+"TM-10051448.jpg")


In [ ]:
second = pd.read_csv("./test_output2.csv")
second[second.filename == "TM-10051448.jpg"]

In [ ]:
import pickle
with open("wtf.pkl", "rb") as handle:
    script_inputs, loaded_files, original_sizes, outputs, detections = pickle.load(handle)

with open("wtf4.pkl", "rb") as handle:
    script_inputs2, loaded_files2, original_sizes2, outputs2, detections2 = pickle.load(handle)

In [ ]:
detections2[1]

In [ ]:
nb_outputs.logits

In [ ]:
script_inputs2.l

In [ ]:
[i.size for i in (test_img, test_img2)]

In [ ]:
original_sizes

In [ ]:
outputs["logits"][1][29, 27]

In [ ]:
nb_outputs["logits"][1][29, 27]

In [ ]:
outputs2["logits"][1][29, 27]

In [ ]:
detections[1]

In [ ]:
outputs.logits[1].numpy().flatten()[2834]

In [ ]:
nb_outputs.logits[1].numpy().flatten()[2834]

In [ ]:
results[1]

In [ ]:
detections[1]

In [ ]:
detections2[1]

In [ ]:
results[1]

In [ ]:
test_img2.convert("RGB")

---
## manual

In [ ]:
def get_labels(batch_logits, threshold):
    batch_class_logits = torch.max(batch_logits, dim=-1)
    batch_scores = torch.sigmoid(batch_class_logits.values)
    batch_labels = batch_class_logits.indices
    results = []
    for scores, labels in zip(batch_scores, batch_labels):
        keep = scores > threshold
        scores = scores[keep]
        labels = labels[keep]
        label_texts = [tags[i] for i in labels]
        results.append({"scores": scores, "labels": labels, 
                        "label_texts": label_texts})
    return results

In [ ]:
get_labels(outputs.logits, threshold=0.2)[1]

In [ ]:
t0 = time()
with torch.inference_mode():
    first_outputs = model(**inputs)

t1 = time()-t0
print(f"that took {t1:.2f}s ({t1/len(imgs_for_proc):.2f}s per image)")

In [ ]:
get_labels(first_outputs.logits, threshold=0.2)[1]